In [1]:
# --- 1. Импорты и общие настройки ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from pmdarima import auto_arima
from statsmodels.tsa.statespace.sarimax import SARIMAX
from prophet import Prophet  # pip install prophet
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings

warnings.filterwarnings("ignore")
# plt.style.use("seaborn-whitegrid")



In [2]:
# --- 2. Загрузка данных ---
# df должен содержать столбцы: "Регион", "Период" (YYYY-MM), и целевой столбец с показателем.
df = pd.read_excel("Датасет по лошадям v2.xlsx")
df["Период"] = pd.to_datetime(df["Период"], format="%Y-%m")
df.sample(10)



,Регион,Период,Лошади
605,ВОСТОЧНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,2023-02-01,825.75
29,АКМОЛИНСКАЯ ОБЛАСТЬ,2017-06-01,1814.28
1262,КАРАГАНДИНСКАЯ ОБЛАСТЬ,2024-09-01,2785.86
431,АТЫРАУСКАЯ ОБЛАСТЬ,2019-03-01,650.14
64,АКМОЛИНСКАЯ ОБЛАСТЬ,2020-05-01,1130.97
372,АЛМАТИНСКАЯ ОБЛАСТЬ,2024-11-01,2563.15
410,АТЫРАУСКАЯ ОБЛАСТЬ,2017-06-01,547.30
202,АКТЮБИНСКАЯ ОБЛАСТЬ,2021-04-01,1391.71
914,ЖАМБЫЛСКАЯ ОБЛАСТЬ,2016-11-01,1300.52
1180,КАРАГАНДИНСКАЯ ОБЛАСТЬ,2017-11-01,1478.60


In [3]:
# === загружаем данные ===
best_methods = pd.read_excel("results/Лошади - Лучшие модели (MAPE_then_MAE) v2.xlsx")  # лучшие методы
best_methods

,Регион,MAPE_HW,MAE_HW,MAPE_SARIMA,MAE_SARIMA,MAPE_Prophet,MAE_Prophet,Best_method,Best_criterion,Best_MAPE,Best_MAE
0,АЛМАТИНСКАЯ ОБЛАСТЬ,8.89,152.38,10.08,163.33,31.63,594.89,HW,MAPE,8.89,152.38
1,ГШЫМКЕНТ,49.18,44.93,91.95,64.55,75.19,60.99,HW,MAPE,49.18,44.93
2,ЖАМБЫЛСКАЯ ОБЛАСТЬ,3.12,42.54,3.74,50.01,6.41,96.40,HW,MAPE,3.12,42.54
3,КОСТАНАЙСКАЯ ОБЛАСТЬ,8.34,29.25,9.30,43.93,8.68,41.29,HW,MAPE,8.34,29.25
4,МАНГИСТАУСКАЯ ОБЛАСТЬ,20.61,49.07,27.49,68.51,25.06,57.53,HW,MAPE,20.61,49.07
5,ОБЛАСТЬ АБАЙ,10.64,408.71,13.06,462.21,22.44,822.99,HW,MAPE,10.64,408.71
6,ОБЛАСТЬ ҰЛЫТАУ,3.95,48.17,8.50,106.31,12.48,187.08,HW,MAPE,3.95,48.17
7,ПАВЛОДАРСКАЯ ОБЛАСТЬ,20.12,483.79,21.84,525.44,23.44,445.61,HW,MAPE,20.12,445.61
8,АКМОЛИНСКАЯ ОБЛАСТЬ,5.75,76.92,5.08,68.16,6.00,79.54,SARIMA,MAPE,5.08,68.16
9,АКТЮБИНСКАЯ ОБЛАСТЬ,9.32,225.85,7.20,169.39,9.90,222.54,SARIMA,MAPE,7.20,169.39


In [4]:
actual_aug = pd.read_excel("Лошади 08.2025.xlsx")
actual_aug["Период"] = pd.to_datetime(actual_aug["Период"], format="%Y-%m")
actual_aug["Лошади"] = (actual_aug["Лошади"]
                     .astype(str)
                     .str.replace(".", "", regex=False)   # убираем разделители тысяч
                     .str.replace(",", ".", regex=False)  # заменяем запятую на точку
                     .astype(float))
actual_aug.to_excel("Лошади обработанные август 2025.xlsx", index=False)
actual_aug



,Регион,Период,Лошади
0,АКМОЛИНСКАЯ ОБЛАСТЬ,2025-08-01,934.00
1,АКТЮБИНСКАЯ ОБЛАСТЬ,2025-08-01,1092.57
2,АЛМАТИНСКАЯ ОБЛАСТЬ,2025-08-01,1730.09
3,АТЫРАУСКАЯ ОБЛАСТЬ,2025-08-01,852.50
4,ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,2025-08-01,668.05
5,ЖАМБЫЛСКАЯ ОБЛАСТЬ,2025-08-01,1790.84
6,КАРАГАНДИНСКАЯ ОБЛАСТЬ,2025-08-01,1329.79
7,КОСТАНАЙСКАЯ ОБЛАСТЬ,2025-08-01,446.02
8,КЫЗЫЛОРДИНСКАЯ ОБЛАСТЬ,2025-08-01,752.27
9,МАНГИСТАУСКАЯ ОБЛАСТЬ,2025-08-01,135.30


In [5]:
# === настройки ===
TARGET = "Лошади"
CUTOFF = "2025-07-01"
FORECAST = "2025-08-01"
EPS = 1e-6
SEAS = 12

In [6]:
# оставляем только август 2025 для проверки
fact_aug = (actual_aug[actual_aug["Период"] == "2025-08-01"]
            .set_index("Регион")[TARGET])


In [7]:
# === функции прогнозов, строго как в обучающем коде ===
def fc_hw_like_training(train):
    # train — Series с MS частотой
    train_log = np.log1p(train)  # log1p
    model = ExponentialSmoothing(train_log, seasonal="add", seasonal_periods=SEAS)\
            .fit(optimized=True)
    fc_log = model.forecast(1)
    return float(np.expm1(fc_log).iloc[0])  # expm1

In [8]:
def fc_sarima_like_training(train):
    train_plus = train + EPS
    train_log  = np.log(train_plus)
    use_seasonal = len(train_log) >= 2 * SEAS

    sar = auto_arima(
        train_log,
        seasonal=use_seasonal,
        m=SEAS if use_seasonal else 1,
        D=1 if use_seasonal else 0,
        seasonal_test=None,
        boxcox=True,            # как в обучении
        stepwise=True,
        suppress_warnings=True,
        error_action="ignore"
    )

    fc_log = sar.predict(n_periods=1)
    # берём первый элемент позиционно, независимо от типа (Series/ndarray/scalar)
    fc_log_scalar = np.asarray(fc_log).ravel()[0]

    return float(np.exp(fc_log_scalar) - EPS)

In [9]:
def fc_prophet_like_training(train):
    df_p = (train.reset_index()
                 .rename(columns={"Период": "ds", TARGET: "y"}))
    df_p["y"] = np.log(df_p["y"] + EPS)       # лог как в обучении
    m = Prophet()
    m.fit(df_p)
    future = m.make_future_dataframe(periods=1, freq="MS")
    yhat_log = m.predict(future)["yhat"].iloc[-1]
    return float(np.exp(yhat_log) - EPS)

In [10]:
methods_map = {
    "HW": fc_hw_like_training,
    "Holt-Winters": fc_hw_like_training,
    "Holt_Winters": fc_hw_like_training,
    "SARIMA": fc_sarima_like_training,
    "Prophet": fc_prophet_like_training,
}

In [12]:
# === прогон по регионам согласно «лучшему методу» ===
rows = []
for _, r in best_methods.iterrows():
    region = r["Регион"]
    method = r["Best_method"]

    ts = (df[df["Регион"] == region]
          .set_index("Период")[TARGET]
          .asfreq("MS")
          .sort_index())

    train = ts[:CUTOFF].dropna()
    if len(train) < 24:
        # как и в обучении, пропускаем короткие ряды
        continue

    # вызов нужной функции
    f = methods_map.get(method)
    if f is None:
        # на всякий случай нормализуем ключи
        key = str(method).strip().upper()
        if key == "HW" or "HOLT" in key:
            f = fc_hw_like_training
        elif "SARIMA" in key or "ARIMA" in key:
            f = fc_sarima_like_training
        else:
            f = fc_prophet_like_training

    fc = f(train)
    actual = fact_aug.get(region, np.nan)
    pct_dev = (fc - actual) / actual * 100 if pd.notna(actual) else np.nan

    rows.append({
        "Регион": region,
        "Лучший метод": method,
        "Прогноз (2025-08)": round(fc, 2),
        "Факт (2025-08)": round(actual, 2) if pd.notna(actual) else np.nan,
        "Отклонение, %": round(pct_dev, 2) if pd.notna(pct_dev) else np.nan
    })

results_aug = pd.DataFrame(rows).sort_values("Регион").reset_index(drop=True)
results_aug.to_excel("results/Лошади - Прогноз на 2025-08 (как в обучении).xlsx")
results_aug

,Регион,Лучший метод,Прогноз (2025-08),Факт (2025-08),"Отклонение, %"
0,АКМОЛИНСКАЯ ОБЛАСТЬ,SARIMA,983.53,934.00,5.30
1,АКТЮБИНСКАЯ ОБЛАСТЬ,SARIMA,1050.98,1092.57,-3.81
2,АЛМАТИНСКАЯ ОБЛАСТЬ,HW,2053.96,1730.09,18.72
3,АТЫРАУСКАЯ ОБЛАСТЬ,SARIMA,715.91,852.50,-16.02
4,ВОСТОЧНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,SARIMA,1852.59,1776.12,4.31
5,ГАСТАНА,SARIMA,2.11,NaN,NaN
6,ГШЫМКЕНТ,HW,56.33,NaN,NaN
7,ЖАМБЫЛСКАЯ ОБЛАСТЬ,HW,1790.32,1790.84,-0.03
8,ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,SARIMA,685.13,668.05,2.56
9,КАРАГАНДИНСКАЯ ОБЛАСТЬ,SARIMA,1334.53,1329.79,0.36
